In [ ]:
from scipy.stats import wilcoxon, ttest_rel
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# === Angular Deviation Data ===
labels_synth = [
    "Clean", "Clean Outliers", "Uniform",
    "Gaussian", "Feature",
    "Uniform+Out", "Gaussian+Out", "Feature+Out"
]

hinge_synth = [0.00, 0.92, 0.65, 5.95, 0.26, 0.98, 6.02, 0.96]
pinball_synth = [0.00, 0.78, 0.56, 5.81, 0.26, 0.70, 6.00, 0.92]
eagle_synth = [0.00, 0.65, 0.53, 4.58, 0.26, 0.65, 4.96, 0.88]
elm_synth = [0.00, 0.98, 2.89, 14.98, 0.60, 3.37, 15.62, 1.40]


labels_keel = ["Heart", "Pima", "Ionosphere", "Sonar"]

# --- Hinge SVM (Table 4)
hinge_keel = [3.76, 9.03, 7.30, 11.98]

# --- Pinball SVM (Table 6)
pinball_keel = [0.00, 10.00, 7.36, 12.36]

# --- Eagle SVM (Table 8)
eagle_keel = [0.00, 5.85, 4.36, 9.96]

# --- ELM (Table 10)
elm_keel = [1.65, 1.58, 2.50, 3.31]


# === Statistical Testing Function ===
def run_tests(x, y, label, test_data):
    t_stat, t_p = ttest_rel(x, y)
    try:
        w_stat, w_p = wilcoxon(x, y, alternative='less')
    except ValueError:
        w_p = None
    test_data.append({
        "Model Pair": label,
        "t-test p-value": round(t_p, 4),
        "Wilcoxon p-value": round(w_p, 4) if w_p is not None else "NA",
        "Significant (α=0.05)": "Yes" if w_p is not None and w_p < 0.05 else "No"
    })

# Run Tests
test_results_synth = []
test_results_keel = []

run_tests(hinge_synth, elm_synth, "Hinge vs ELM", test_results_synth)
run_tests(pinball_synth, elm_synth, "Pinball vs ELM", test_results_synth)
run_tests(eagle_synth, elm_synth, "Eagle vs ELM", test_results_synth)

run_tests(hinge_keel, elm_keel, "Hinge vs ELM", test_results_keel)
run_tests(pinball_keel, elm_keel, "Pinball vs ELM", test_results_keel)
run_tests(eagle_keel, elm_keel, "Eagle vs ELM", test_results_keel)

# Export to CSV
df_synth = pd.DataFrame(test_results_synth)
df_keel = pd.DataFrame(test_results_keel)

df_synth.to_csv("hypothesis_results_synthetic.csv", index=False)
df_keel.to_csv("hypothesis_results_keel.csv", index=False)

# === Prepare Data for Plotting ===
def prepare_plot_data(values_dict, labels):
    rows = []
    for model, values in values_dict.items():
        for lbl, v in zip(labels, values):
            rows.append({"Noise Type": lbl, "Model": model, "Angular Deviation": v})
    return pd.DataFrame(rows)

# Synthetic and KEEL values in dicts
synth_dict = {
    "Hinge": hinge_synth,
    "Pinball": pinball_synth,
    "Eagle": eagle_synth,
    "ELM": elm_synth
}

keel_dict = {
    "Hinge": hinge_keel,
    "Pinball": pinball_keel,
    "Eagle": eagle_keel,
    "ELM": elm_keel
}

df_synth_plot = prepare_plot_data(synth_dict, labels_synth)
df_keel_plot = prepare_plot_data(keel_dict, labels_keel)

# === Plotting ===
def plot_violin_box(df, title_prefix, save_prefix):
    plt.figure(figsize=(10, 6))
    sns.violinplot(data=df, x="Model", y="Angular Deviation", inner="box")
    plt.title(f"{title_prefix} - Violin Plot")
    plt.savefig(f"{save_prefix}_violin.png", dpi=300)
    plt.clf()

    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x="Model", y="Angular Deviation")
    plt.title(f"{title_prefix} - Box Plot")
    plt.savefig(f"{save_prefix}_box.png", dpi=300)
    plt.clf()

plot_violin_box(df_synth_plot, "Synthetic Angular Deviation", "synthetic_ad")
plot_violin_box(df_keel_plot, "KEEL Angular Deviation", "keel_ad")

# === Print Results ===
print("=== Statistical Results: Synthetic ===")
print(df_synth)
print("\n=== Statistical Results: KEEL ===")
print(df_keel)


=== Statistical Results: Synthetic ===
       Model Pair  t-test p-value  Wilcoxon p-value Significant (α=0.05)
0    Hinge vs ELM          0.0708            0.0078                  Yes
1  Pinball vs ELM          0.0650            0.0078                  Yes
2    Eagle vs ELM          0.0690            0.0078                  Yes

=== Statistical Results: KEEL ===
       Model Pair  t-test p-value  Wilcoxon p-value Significant (α=0.05)
0    Hinge vs ELM          0.0291            1.0000                   No
1  Pinball vs ELM          0.1257            0.9375                   No
2    Eagle vs ELM          0.2143            0.9375                   No


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>